# Translation using Transformers

Transformers have become the defacto standard for NLP tasks nowadays. They started being used in NLP but they are now being used in Computer Vision and sometimes to generate music as well. I am sure you would all have heard about the GPT3 Transformer or the jokes thereof.

But everything aside, they are still hard to understand as ever. In my last post, I talked in quite a detail about transformers and how they work on a basic level. I went through the encoder and decoder architecture and the whole data flow in those different pieces of the neural network.

But as I like to say we don't really understand something before we implement it ourselves. So in this post, we will implement an English to German language translator using Transformers.

# Task Description

We want to create a translator that uses transformers to convert English to German. So, if we look at it as a black-box, our network takes as input an English sentence and returns a German sentence.

In [ ]:
from typing import Optional
from collections import Counter
import torch
from torch.nn.init import xavier_uniform_
from torch import Tensor
import torch.nn as nn
import torch.nn.functional as F
import math
import spacy
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import plotly.express as px
import pandas as pd

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f"Using device: {device}")

# Data Preprocessing

To create a translation Model, we need translated sentence pairs between English and German. We use the `Helsinki-NLP/opus-100` dataset (German-English subset), which is a curated multilingual translation corpus available via HuggingFace `datasets`.

Load the spaCy models — these will be used for tokenization of German and English text.

In [ ]:
!python -m spacy download de_core_news_sm
!python -m spacy download en_core_web_sm

In [ ]:
# Load the spaCy models
spacy_de = spacy.load('de_core_news_sm')
spacy_en = spacy.load('en_core_web_sm')

def tokenize_de(text):
    return [tok.text for tok in spacy_de.tokenizer(text)]

def tokenize_en(text):
    return [tok.text for tok in spacy_en.tokenizer(text)]

Define some special tokens we will use for specifying blank/padding words, and beginning and end of sentences.

In [ ]:
# Special Tokens
BOS_WORD = '<s>'
EOS_WORD = '</s>'
BLANK_WORD = "<blank>"

We load the OPUS-100 de-en dataset using HuggingFace `datasets` and filter sentences by length.

In [ ]:
MAX_LEN = 20
MAX_TRAIN_EXAMPLES = 200_000  # Cap training set; opus-100 has ~1M pairs

raw_dataset = load_dataset("Helsinki-NLP/opus-100", "de-en")
train_data = raw_dataset["train"]
val_data = raw_dataset["validation"]
test_data = raw_dataset["test"]

# Filter to sentences within MAX_LEN tokens
def filter_by_length(example):
    en_tokens = tokenize_en(example["translation"]["en"])
    de_tokens = tokenize_de(example["translation"]["de"])
    return len(en_tokens) <= MAX_LEN and len(de_tokens) <= MAX_LEN

train_data = train_data.filter(filter_by_length)
val_data = val_data.filter(filter_by_length)

# Cap training size to keep memory and runtime reasonable
if len(train_data) > MAX_TRAIN_EXAMPLES:
    train_data = train_data.select(range(MAX_TRAIN_EXAMPLES))

print(f"Train examples: {len(train_data)}, Val examples: {len(val_data)}")

Let us try to see what our data looks like:

In [ ]:
for i, example in enumerate(train_data.select(range(5))):
    en = tokenize_en(example["translation"]["en"])
    de = tokenize_de(example["translation"]["de"])
    print(f"Example_{i}: ({en}, {de})")

Now we build the source and target vocabularies from token frequency counts. We specify a `MIN_FREQ` of 2 so that any word that doesn't occur at least twice doesn't get to be part of our vocabulary.

In [ ]:
MIN_FREQ = 2

class Vocab:
    """Minimal vocabulary: token-to-index and index-to-token mappings."""
    def __init__(self, counter, min_freq=1, specials=None):
        self._itos = list(specials or [])
        self._stoi = {tok: i for i, tok in enumerate(self._itos)}
        for tok, freq in counter.most_common():
            if freq < min_freq:
                break
            if tok not in self._stoi:
                self._stoi[tok] = len(self._itos)
                self._itos.append(tok)
        self._default_index = 0

    def set_default_index(self, index):
        self._default_index = index

    def __getitem__(self, token):
        return self._stoi.get(token, self._default_index)

    def __len__(self):
        return len(self._itos)

    def __call__(self, tokens):
        return [self[t] for t in tokens]

    def get_itos(self):
        return self._itos

# Count token frequencies
src_counter = Counter()
tgt_counter = Counter()
for example in train_data:
    src_counter.update(tokenize_en(example["translation"]["en"]))
    tgt_counter.update(tokenize_de(example["translation"]["de"]))

# Build vocabularies
SRC_VOCAB = Vocab(src_counter, min_freq=MIN_FREQ, specials=[BLANK_WORD, BOS_WORD, EOS_WORD])
SRC_VOCAB.set_default_index(SRC_VOCAB[BLANK_WORD])

TGT_VOCAB = Vocab(tgt_counter, min_freq=MIN_FREQ, specials=[BLANK_WORD, BOS_WORD, EOS_WORD])
TGT_VOCAB.set_default_index(TGT_VOCAB[BLANK_WORD])

print(f"Source vocab size: {len(SRC_VOCAB)}, Target vocab size: {len(TGT_VOCAB)}")

Now we create a `collate_fn` that tokenizes, numericalizes, and pads each batch, and wrap everything in a `DataLoader`.

In [ ]:
PAD_IDX = SRC_VOCAB[BLANK_WORD]
BOS_IDX = TGT_VOCAB[BOS_WORD]
EOS_IDX = TGT_VOCAB[EOS_WORD]

def collate_fn(batch):
    src_batch, tgt_batch = [], []
    for example in batch:
        src_tokens = tokenize_en(example["translation"]["en"])
        tgt_tokens = tokenize_de(example["translation"]["de"])
        src_indices = torch.tensor(SRC_VOCAB(src_tokens), dtype=torch.long)
        # Prepend BOS and append EOS to target
        tgt_indices = torch.tensor(
            [BOS_IDX] + TGT_VOCAB(tgt_tokens) + [EOS_IDX], dtype=torch.long
        )
        src_batch.append(src_indices)
        tgt_batch.append(tgt_indices)
    # Pad sequences — shape: (max_seq_len, batch_size)
    src_batch = pad_sequence(src_batch, padding_value=PAD_IDX)
    tgt_batch = pad_sequence(tgt_batch, padding_value=PAD_IDX)
    return src_batch, tgt_batch

We create `DataLoader` instances for training and validation. The training loader shuffles and batches data, while the validation loader uses a batch size of 1 to minimize padding when checking performance.

In [ ]:
BATCH_SIZE = 64

train_iter = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_iter = DataLoader(val_data, batch_size=1, shuffle=False, collate_fn=collate_fn)

We can see what's in a batch. And what we are sending to the model as an input while training.

In [ ]:
src_batch, tgt_batch = next(iter(train_iter))
# src_batch shape: (max_src_len, batch_size), tgt_batch shape: (max_tgt_len, batch_size)
src_matrix = src_batch.T
print(src_matrix, src_matrix.size())

In [ ]:
trg_matrix = tgt_batch.T
print(trg_matrix, trg_matrix.size())

So the src tensor contains sentences padded to the same length and the target tensor similarly. The numbers represent token indices in the vocabulary. For example, the padding token index in the source vocabulary:

In [ ]:
print(SRC_VOCAB.get_itos()[PAD_IDX])   # padding token
print(TGT_VOCAB.get_itos()[BOS_IDX])   # start token
print(TGT_VOCAB.get_itos()[PAD_IDX])   # padding token

And what about the end-of-sentence token?

In [ ]:
TGT_VOCAB[EOS_WORD]

**Important:** It doesn't matter if you use a different preprocessing pipeline or tokenizer. What matters is that in the end you send the source and target sentences to your model in the format expected by the Transformer: source sentences padded with a blank token, and target sentences with a start token, an end token, and padding.

So now that we have a way to send the input sentence and the shifted outputs to our transformer, we can look at creating the Transformer ourselves. A lot of the blocks here are taken from the PyTorch `nn` module. In fact, PyTorch has a Transformer module too but it doesn't include a lot of functionalities present in the paper like the embedding layer, and the PositionalEncoding layer. So this is sort of a more complete implementation.

We create our Transformer particularly using these various blocks from the PyTorch `nn` module:

- [TransformerEncoderLayer](https://pytorch.org/docs/master/generated/torch.nn.TransformerEncoderLayer.html) :  A single encoder layer
- [TransformerEncoder](https://pytorch.org/docs/stable/generated/torch.nn.TransformerEncoder.html) : A stack of `num_encoder_layers` layers. In the paper it is by default kept as 6.
- [TransformerDecoderLayer](https://pytorch.org/docs/stable/generated/torch.nn.TransformerDecoderLayer.html) : A single decoder layer
- [TransformerDecoder](https://pytorch.org/docs/stable/generated/torch.nn.TransformerDecoder.html) : A stack of `num_decoder_layers` layers. In the paper it is by default kept as 6.

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x * math.sqrt(self.d_model)
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)


class MyTransformer(nn.Module):
    def __init__(self, d_model: int = 512, nhead: int = 8, num_encoder_layers: int = 6,
                 num_decoder_layers: int = 6, dim_feedforward: int = 2048, dropout: float = 0.1,
                 activation: str = "relu", source_vocab_length: int = 60000,
                 target_vocab_length: int = 60000) -> None:
        super(MyTransformer, self).__init__()
        self.source_embedding = nn.Embedding(source_vocab_length, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, activation)
        encoder_norm = nn.LayerNorm(d_model)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_encoder_layers, encoder_norm)
        self.target_embedding = nn.Embedding(target_vocab_length, d_model)
        decoder_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout, activation)
        decoder_norm = nn.LayerNorm(d_model)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_decoder_layers, decoder_norm)
        self.out = nn.Linear(d_model, target_vocab_length)
        self._reset_parameters()
        self.d_model = d_model
        self.nhead = nhead

    def forward(self, src: Tensor, tgt: Tensor, src_mask: Optional[Tensor] = None, tgt_mask: Optional[Tensor] = None,
                memory_mask: Optional[Tensor] = None, src_key_padding_mask: Optional[Tensor] = None,
                tgt_key_padding_mask: Optional[Tensor] = None, memory_key_padding_mask: Optional[Tensor] = None) -> Tensor:
        if src.size(1) != tgt.size(1):
            raise RuntimeError("the batch number of src and tgt must be equal")
        src = self.source_embedding(src)
        src = self.pos_encoder(src)
        memory = self.encoder(src, mask=src_mask, src_key_padding_mask=src_key_padding_mask)
        tgt = self.target_embedding(tgt)
        tgt = self.pos_encoder(tgt)
        output = self.decoder(tgt, memory, tgt_mask=tgt_mask, memory_mask=memory_mask,
                              tgt_key_padding_mask=tgt_key_padding_mask,
                              memory_key_padding_mask=memory_key_padding_mask)
        output = self.out(output)
        return output

    def _reset_parameters(self):
        r"""Initiate parameters in the transformer model."""
        for p in self.parameters():
            if p.dim() > 1:
                xavier_uniform_(p)

Now, we can initialize the transformer and the optimizer using:

In [ ]:
source_vocab_length = len(SRC_VOCAB)
target_vocab_length = len(TGT_VOCAB)

model = MyTransformer(source_vocab_length=source_vocab_length, target_vocab_length=target_vocab_length)
optim = torch.optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
model = model.to(device)

We can train using the simple train loop

In [ ]:
def make_pad_mask(seq, pad_idx):
    """Build a float padding mask: 0.0 for real tokens, -inf for padding."""
    mask = (seq == pad_idx).float()
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

def make_causal_mask(size, device):
    """Build a float causal mask: 0.0 below the diagonal, -inf above."""
    mask = torch.triu(torch.ones(size, size, device=device), diagonal=1)
    mask = mask.masked_fill(mask == 1, float('-inf'))
    return mask

def train(train_iter, val_iter, model, optim, num_epochs, device):
    train_losses = []
    valid_losses = []
    for epoch in range(num_epochs):
        train_loss = 0
        valid_loss = 0

        # Train model
        model.train()
        for i, (src, trg) in enumerate(train_iter):
            # src shape: (src_len, batch_size), trg shape: (trg_len, batch_size)
            src = src.to(device)
            trg = trg.to(device)

            # Transpose to (batch_size, seq_len) for slicing
            src = src.transpose(0, 1)
            trg = trg.transpose(0, 1)

            trg_input = trg[:, :-1]
            targets = trg[:, 1:].contiguous().view(-1)

            # Float masks (same dtype as causal mask, avoids dtype-mismatch warning)
            src_key_padding_mask = make_pad_mask(src, PAD_IDX).to(device)
            tgt_key_padding_mask = make_pad_mask(trg_input, PAD_IDX).to(device)
            np_mask = make_causal_mask(trg_input.size(1), device)

            # Forward, backprop, optimizer
            optim.zero_grad()
            preds = model(
                src.transpose(0, 1), trg_input.transpose(0, 1),
                tgt_mask=np_mask,
                src_key_padding_mask=src_key_padding_mask,
                tgt_key_padding_mask=tgt_key_padding_mask,
                memory_key_padding_mask=src_key_padding_mask,
            )
            preds = preds.transpose(0, 1).contiguous().view(-1, preds.size(-1))
            loss = F.cross_entropy(preds, targets, ignore_index=PAD_IDX, reduction='sum')
            loss.backward()
            optim.step()
            train_loss += loss.item() / BATCH_SIZE

        model.eval()
        with torch.no_grad():
            for i, (src, trg) in enumerate(val_iter):
                src = src.to(device)
                trg = trg.to(device)

                src = src.transpose(0, 1)
                trg = trg.transpose(0, 1)

                trg_input = trg[:, :-1]
                targets = trg[:, 1:].contiguous().view(-1)

                src_key_padding_mask = make_pad_mask(src, PAD_IDX).to(device)
                tgt_key_padding_mask = make_pad_mask(trg_input, PAD_IDX).to(device)
                np_mask = make_causal_mask(trg_input.size(1), device)

                preds = model(
                    src.transpose(0, 1), trg_input.transpose(0, 1),
                    tgt_mask=np_mask,
                    src_key_padding_mask=src_key_padding_mask,
                    tgt_key_padding_mask=tgt_key_padding_mask,
                    memory_key_padding_mask=src_key_padding_mask,
                )
                preds = preds.transpose(0, 1).contiguous().view(-1, preds.size(-1))
                loss = F.cross_entropy(preds, targets, ignore_index=PAD_IDX, reduction='sum')
                valid_loss += loss.item() / 1

        # Log after each epoch
        print(f"Epoch [{epoch+1}/{num_epochs}] complete. "
              f"Train Loss: {train_loss/len(train_iter):.3f}. "
              f"Val Loss: {valid_loss/len(val_iter):.3f}")

        # Save best model
        if valid_loss / len(val_iter) < min(valid_losses, default=1e9):
            print("saving state dict")
            torch.save(model.state_dict(), "checkpoint_best_epoch.pt")

        train_losses.append(train_loss / len(train_iter))
        valid_losses.append(valid_loss / len(val_iter))

        # Check example after each epoch
        sentences = ["This is an example to check how our model is performing."]
        for sentence in sentences:
            print(f"Original Sentence: {sentence}")
            print(f"Translated Sentence: {greedy_decode_sentence(model, sentence)}")

    return train_losses, valid_losses

In [ ]:
def greedy_decode_sentence(model, sentence):
    model.eval()
    tokens = tokenize_en(sentence)
    indexed = SRC_VOCAB(tokens)
    sentence_tensor = torch.LongTensor([indexed]).to(device)

    trg_init_tok = TGT_VOCAB[BOS_WORD]
    trg = torch.LongTensor([[trg_init_tok]]).to(device)
    translated_sentence = ""
    maxlen = 25

    for i in range(maxlen):
        np_mask = make_causal_mask(trg.size(0), device)
        pred = model(sentence_tensor.transpose(0, 1), trg, tgt_mask=np_mask)
        add_word = TGT_VOCAB.get_itos()[pred.argmax(dim=2)[-1].item()]
        translated_sentence += " " + add_word
        if add_word == EOS_WORD:
            break
        trg = torch.cat((trg, torch.LongTensor([[pred.argmax(dim=2)[-1].item()]]).to(device)))

    return translated_sentence

In [ ]:
train_losses, valid_losses = train(train_iter, val_iter, model, optim, 35, device)

In [ ]:
losses = pd.DataFrame({'train_loss': train_losses, 'val_loss': valid_losses})

In [ ]:
losses.head()

In [ ]:
px.line(losses, y=['train_loss', 'val_loss'])

We can load the best model using the saved checkpoint.

In [ ]:
# Load the best model for inference
model.load_state_dict(torch.load("checkpoint_best_epoch.pt", map_location=device, weights_only=True))

Let us see the output on a single sentence

In [ ]:
sentence = "Isn't Natural language processing just awesome? Please do let me know in the comments."
print(greedy_decode_sentence(model, sentence))